# Training the 915M model on Kaggle

Sized for the box you have: **T4 x2 (15 GB each), 30 GB RAM, 12-hour sessions.**

Before running, in the sidebar set:

- **Accelerator → GPU T4 x2**
- **Internet → On** (needed to clone the repo and download the corpus)

### Why this configuration

The model is 914,729,472 parameters. What decides whether it fits on a 15 GB
card is the optimizer, because AdamW keeps two fp32 moments per parameter:

```
AdamW   weights 3.7 + grads 3.7 + m 3.7 + v 3.7 = 14.6 GB  ->  will not fit
SGD     weights 3.7 + grads 3.7               =  7.3 GB  ->  fits
```

So: **SGD, gradient checkpointing, fp16 mixed precision.** T4s have tensor
cores for fp16 but no bf16 (that needs Ampere), which is why `--dtype float16`
rather than the default bfloat16.

**One T4 is used, not both.** This project has no data-parallel support — no
DDP, no FSDP — so the second card sits idle. Adding it is real work, not a flag.


## 1. Confirm the hardware


In [ ]:
import subprocess, torch, os
print(subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| devices', torch.cuda.device_count())
print('free disk:', round(os.statvfs('/kaggle/working').f_bavail *
                          os.statvfs('/kaggle/working').f_frsize / 1e9, 1), 'GB')


## 2. Code and dependencies


In [ ]:
%cd /kaggle/working
!git clone --branch main-5h8bvh --single-branch \
    https://github.com/m9cherif/ai_from_zero.git 2>/dev/null || echo 'already cloned'
%cd /kaggle/working/ai_from_zero
!pip install -q -r requirements.txt

from myai.core.device import setup, describe
print(describe(setup('auto')))


## 3. Corpus and tokenizer

About 25 million tokens. Skip this cell if you attached a previous run's output
as a dataset — see the last section on resuming.


In [ ]:
!python scripts/fetch_corpus.py
!python scripts/build_tokenizer.py --type bpe --vocab-size 4096 \
    --data 'data/train/*.txt'


## 4. Measure the speed before choosing a step budget

Run 60 steps and read the reported `tokens/s`. Everything else follows from it —
guessing here is how people discover at hour 11 that they set the schedule wrong.

The learning-rate schedule is a cosine over `--steps`, so that number needs to be
roughly right from the start: set it too high and the run ends with the rate still
large; too low and it anneals to nothing while time remains.


In [ ]:
!python scripts/train.py --preset xl \
    --data data/train --cache-dir output/tokens \
    --steps 60 --batch-size 8 --lr 0.02 \
    --optimizer sgd --grad-checkpoint --no-save-optimizer \
    --dtype float16 --mixed-precision --flash --device auto \
    --log-every 10 --save-every 100000 --output /kaggle/working/probe


### Turn that into a step count

Put the measured tokens/s into the cell below. It leaves an hour of the 12 for
setup, evaluation and saving — a run killed by the session clock loses whatever
it had not written to disk.


In [ ]:
MEASURED_TOKENS_PER_SEC = 1500   # <- replace with the number the probe printed
BATCH, SEQ = 8, 256
HOURS = 10.5

tokens_per_step = BATCH * SEQ
steps = int(MEASURED_TOKENS_PER_SEC * HOURS * 3600 / tokens_per_step)
tokens = steps * tokens_per_step
print(f'steps for {HOURS} h : {steps:,}')
print(f'tokens seen      : {tokens/1e6:,.1f}M  ({tokens/24_866_609:.2f} epochs)')
print(f'tokens/parameter : {tokens/914_729_472:.3f}   (Chinchilla-optimal is ~20)')


## 5. Train

Checkpoints land in `/kaggle/working/xl`, which becomes the notebook's output and
survives the session. `--no-save-optimizer` keeps each file at 3.5 GB; with SGD at
momentum 0 there is no optimizer state to lose, so resuming stays exact.

Set `--steps` from the cell above. Save often — Kaggle stops the session at 12
hours whether or not the run is finished.


In [ ]:
STEPS = 29000        # <- from the cell above

!python scripts/train.py --preset xl \
    --data data/train --val-data data/val --cache-dir output/tokens \
    --steps {STEPS} --batch-size 8 --lr 0.02 --dropout 0.0 \
    --optimizer sgd --grad-checkpoint --no-save-optimizer \
    --dtype float16 --mixed-precision --flash --device auto \
    --log-every 100 --save-every 2000 \
    --output /kaggle/working/xl


**If it reports out of memory:** drop `--batch-size` to 4, then 2. Gradient
checkpointing is already on, so batch size is the remaining lever. Watch the
first few hundred steps — if VRAM sits well under 15 GB you can raise the batch
to 16 instead, which is faster per token.


## 6. Score and sample


In [ ]:
!python scripts/evaluate.py --data data/val --max-batches 200 \
    --checkpoint /kaggle/working/xl/checkpoint_latest.pt --device auto


In [ ]:
!python scripts/chat.py --max-tokens 60 --device auto \
    --checkpoint /kaggle/working/xl/checkpoint_latest.pt --prompt \
    'The old man walked into the' 'She said that' 'It was the best of'


## 7. Continuing in the next session

Twelve hours will not finish this model, so plan on a chain of sessions:

1. **Save Version** (Save & Run All, or Quick Save) — the contents of
   `/kaggle/working` become the notebook's output. The checkpoint is 3.5 GB
   against Kaggle's 20 GB output limit, so keep one or two, not ten.
2. In the next session, **+ Add Input → Notebook Output** and pick that run.
   It mounts read-only under `/kaggle/input/<name>/`.
3. Point the training cell at it and raise `--steps` to the new cumulative total
   (the step counter resumes where it left off, so 29000 → 58000 for another
   full session).

Copy the checkpoint out of `/kaggle/input` first — that mount is read-only, and
the trainer needs somewhere writable for new checkpoints.


In [ ]:
# In a follow-up session, before training:
# !mkdir -p /kaggle/working/xl
# !cp /kaggle/input/<previous-notebook-output>/xl/checkpoint_latest.pt \
#     /kaggle/working/xl/
#
# then add to the training command:
#     --resume /kaggle/working/xl/checkpoint_latest.pt


## What to expect

Ten hours on a T4 is roughly 55M tokens — about **0.06 tokens per parameter**,
against the ~20 that would train this size properly. That is 35 times more
training than the model has had so far, and the loss should finally move: it has
been pinned at 6.93 across eight measurements spanning a sevenfold range of
training, because 840 steps taught it nothing beyond which words are common.

It will not become the better model, though. On the same corpus an 8.3M model
reached held-out perplexity **47.8** and writes real sentences; the 915M reached
**1,022.8**. Filling a billion parameters needs roughly 18 billion tokens and this
corpus has 25 million, so the ceiling is the data, not the hardware or the hours.

Numbers to watch in the logs — if loss drops below **6.9** in the first thousand
steps, the extra compute is doing something the CPU runs never could.

---

**Repository:** [m9cherif/ai_from_zero](https://github.com/m9cherif/ai_from_zero)
(branch `main-5h8bvh`) · [`docs/SCALING.md`](https://github.com/m9cherif/ai_from_zero/blob/main-5h8bvh/docs/SCALING.md)
has the measured basis for every figure quoted here.
